# Título épico (nose q poner todavía)
> **Integrantes:** Matías Soto, Matías Toledo

Esta tarea trata sobre blablabla

### Librerías importantes para esta tarea

In [1]:
import torch.nn as nn
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [2]:
data = pd.read_csv("Pokemon.csv")
data.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


### Explicacion del dataset

El dataset contiene variables numericas de estadisticas de los Pokemon, como `HP` (puntos de salud), `Attack` (danio de movimientos fisicos), `Defense` (resistencia ante movimientos fisicos), `Sp. Atk` (ataque especial), `Sp. Def` (defensa especial) y `Speed` (velocidad).

Ademas incluye variables categoricas como `Type 1` y `Type 2` (tipo elemental), `Generation` (generacion a la que pertenece) y la variable objetivo `Legendary` (si es legendario o no).

Para este trabajo se considera:

- La columna `#` solo enumera y no aporta informacion.
- La columna `Name` es un identificador unico y no aporta informacion.
- Las variables numericas se mantienen sin cambios.
- `Type 1` y `Type 2` se codifican numericamente.
- `Generation` se mantiene para evaluar su aporte al modelo.
- La variable objetivo `Legendary` se transforma a binaria (0/1).

### Preparacion del dataset

1. Se elimina `Name` por ser un identificador sin valor predictivo.
2. Se convierten `Type 1` y `Type 2` a codigos numericos (en `Type 2`, `NaN` se reemplaza por `None`).
3. Se transforma `Legendary` a 0/1 para clasificacion binaria.
4. Se mantiene `Generation` para evaluar su desempeno en el modelo.
5. Se divide el dataset en entrenamiento y prueba antes de normalizar.
6. Se ajusta el `scaler` solo con entrenamiento y se aplica a prueba para evitar data leakage.

Luego se normalizan las variables para que queden en el rango [0, 1].

In [3]:
df = data.copy()

# Variable objetivo binaria
df["Legendary"] = df["Legendary"].astype(int)

# Definir categorias para Type 1 y Type 2
type1_cat = pd.Categorical(df["Type 1"])
type1_categories = list(type1_cat.categories)
type_categories = type1_categories + ["None"]

# Codificacion de Type 1 y mapeo codigo->categoria
print("Type 1 mapeo codigo->categoria:", dict(enumerate(type_categories)))
df["Type 1"] = pd.Categorical(df["Type 1"], categories=type_categories).codes

# Relleno de los faltantes en Type 2 (NaN) por None
type2 = df["Type 2"].fillna("None")

# Codificacion de Type 2 y mapeo codigo->categoria
print("Type 2 mapeo codigo->categoria:", dict(enumerate(type_categories)))
df["Type 2"] = pd.Categorical(type2, categories=type_categories).codes


Type 1 mapeo codigo->categoria: {0: 'Bug', 1: 'Dark', 2: 'Dragon', 3: 'Electric', 4: 'Fairy', 5: 'Fighting', 6: 'Fire', 7: 'Flying', 8: 'Ghost', 9: 'Grass', 10: 'Ground', 11: 'Ice', 12: 'Normal', 13: 'Poison', 14: 'Psychic', 15: 'Rock', 16: 'Steel', 17: 'Water', 18: 'None'}
Type 2 mapeo codigo->categoria: {0: 'Bug', 1: 'Dark', 2: 'Dragon', 3: 'Electric', 4: 'Fairy', 5: 'Fighting', 6: 'Fire', 7: 'Flying', 8: 'Ghost', 9: 'Grass', 10: 'Ground', 11: 'Ice', 12: 'Normal', 13: 'Poison', 14: 'Psychic', 15: 'Rock', 16: 'Steel', 17: 'Water', 18: 'None'}


In [4]:
# Separación de variables y objetivo
X = df.drop(columns=["#","Name", "Legendary"])
y = df["Legendary"]

X.head()

,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
0,9,13,318,45,49,49,65,65,45,1
1,9,13,405,60,62,63,80,80,60,1
2,9,13,525,80,82,83,100,100,80,1
3,9,13,625,80,100,123,122,120,80,1
4,6,18,309,39,52,43,60,50,65,1


In [ ]:
# Split primero para evitar data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X.values,
    y.values,
    test_size=0.2,
    random_state=42,
    stratify=y.values
 )

# Normalizacion: ajustar en train y aplicar en test
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Tensores para PyTorch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

### Definicion del modelo

Se utiliza un perceptron multicapa con una capa oculta y activacion sigmoide.

- `input_dim = 10`: numero de variables de entrada.
- `hidden_dim = 8`: neuronas en la capa oculta (ajustable).
- `output_dim = 2`: dos clases (legendario y no legendario).

In [6]:
class MultiLayerPerceptron(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim): 
        super(type(self), self).__init__()  
        # Capa oculta
        self.hidden = nn.Linear(input_dim, hidden_dim)
        # Capa de salida
        self.output = nn.Linear(hidden_dim, output_dim) 
        # Función de activación sigmoide   
        self.activation = nn.Sigmoid()
        
    def forward(self, x):
        # Conexión 
        x = self.activation(self.hidden(x))
        return self.output(x)
    
# input_dim = 10, correspondiente a las 10 variables a trabajar
# hidden_dim = 8, ajustable, 8 para empezar
# output_dim = 2, clasificamos 2 categorias, "legendario" y "no legendario"
modelo = MultiLayerPerceptron(input_dim= 10, hidden_dim= 8, output_dim= 2)

### Definicion de optimizador y funcion de costo

#### Optimizador

Se usa `Adam` (descenso de gradiente con tasa adaptativa):

- `lr`: tasa de aprendizaje.
- `betas`: coeficientes para los momentos.
- `weight_decay`: regularizacion L2 para reducir sobreajuste.

#### Funcion de costo

Se usa `CrossEntropyLoss`, adecuada para clasificacion multiclase con salidas sin activar (logits).

La perdida para una muestra se expresa como:

$$
\mathcal{L}(y, \hat y) = -\sum_{c=1}^{C} y_c \log(\hat y_c)
$$

donde $y_c$ es la etiqueta verdadera (one-hot) y $\hat y_c$ es la probabilidad predicha por el modelo para la clase $c$.

In [ ]:
# Optimizador
optimizer = torch.optim.Adam(modelo.parameters(), lr=1e-3, betas=(0.9, 0.999), weight_decay=0)

# Funcion de costo
criterion = torch.nn.CrossEntropyLoss(reduction='mean')

### Entrenamiento del modelo

Se entrena el modelo minimizando la funcion de costo en el conjunto de entrenamiento, actualizando los parametros con `optimizer.step()` en cada epoca.

In [10]:
modelo

MultiLayerPerceptron(
  (hidden): Linear(in_features=10, out_features=8, bias=True)
  (output): Linear(in_features=8, out_features=2, bias=True)
  (activation): Sigmoid()
)

In [11]:
# Forward pass para calcular la loss
logits = modelo(X_train_tensor)
loss = criterion(logits, y_train_tensor)
loss

tensor(1.1589, grad_fn=<NllLossBackward0>)

### Evaluacion del modelo

Se evalua el rendimiento en el conjunto de prueba usando metricas como exactitud y matriz de confusion.

In [ ]:
modelo.evaluar(1.0)

### Preguntas finales
1. Sobre la matriz de confusión, interprete los resultados obtenidos. Con sus palabras defina que significa cada tipo de error. ¿Elegiría a Pokémon ubicados en FP o FN para su equipo?
2. Busque un caso mal clasificado por el modelo, e interprete por qué cree que el modelo se equivocó en ese caso.
3. ¿Cúal fue el mayor desafío que enfrentó al realizar esta tarea? ¿Cómo lo solucionó?


### IA Generativa
1. ¿Utilizó alguna herramienta de IA Generativa para realizar esta tarea? En caso afirmativo, indique cuál o cuáles herramientas utilizó.
2. ¿En qué parte o partes de la tarea utilizó estas herramientas?